# Week 3 · Day 2 — Aggregations: turn rows into a report (`GROUP BY`)

*Yesterday you picked rows. Today you summarize them — one number per group.*

**By the end you'll have shipped:** a **revenue-by-category report** written in SQL — `COUNT`, `SUM`, `AVG`, `GROUP BY`, and `HAVING` — the twin of pandas `groupby`, run through the same `run_sql` helper on Snowflake or DuckDB.

### 📋 Lesson card

| | |
|---|---|
| **Module** | M1b · SQL foundations → M4 · Snowflake (Week 3) |
| **Prerequisites** | W3D1 (`SELECT` / `WHERE` / `ORDER BY`), Week 2 Day 1 (`groupby`) |
| **Est. time** | ~30 min |
| **Capstone slice** | *Query* stored matters — billings by practice area, the reports a partner actually asks for |
| **Difficulty** | Core + `Go Deeper 🔧` |
| **Runs offline?** | ✅ Yes — Snowflake if credentials exist, else DuckDB |

### 🎯 Learning objectives

By the end you'll be able to:
- Use the **aggregate functions** `COUNT`, `SUM`, `AVG`, `MIN`, `MAX` over a whole column.
- **`GROUP BY`** a column to get one summary row per category — the SQL twin of pandas `groupby`.
- Name results with **`AS`** and tidy numbers with **`ROUND`**.
- **`HAVING`** — filter *groups* by their aggregate (and know how it differs from `WHERE`).
- Combine `WHERE` + `GROUP BY` + `ORDER BY` into one real report.

### ⚖️ Why it matters

"How much did we bill per practice area last quarter?" "How many active matters per attorney?" Every one of those is a **group-and-count** question. In a spreadsheet it's a pivot table; in SQL it's `GROUP BY`. This is the single most-used pattern in analytics — learn it on coffee sales today, point it at `matters` tomorrow.

### ⚙️ Setup

Same `run_sql(...)` helper as yesterday — loads the `coffee_orders` table into Snowflake or DuckDB and returns results as a DataFrame.

> 🔒 *Synthetic data only — this coffee set (and the matters set) is fake. Never load real client or privileged data into a teaching notebook.*

In [ ]:
import os, warnings
warnings.filterwarnings("ignore")
import pandas as pd

# Load .env if python-dotenv is present (optional — the notebook runs fine without it).
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# --- Pick the backend: Snowflake if credentials exist in .env, else local DuckDB ---
# You write the SAME SQL either way; the backend is invisible.
SNOWFLAKE_READY = all(os.environ.get(k) for k in ("SNOWFLAKE_ACCOUNT", "SNOWFLAKE_USER", "SNOWFLAKE_PASSWORD"))
BACKEND = "snowflake" if SNOWFLAKE_READY else "duckdb"

def _find(fname):
    for base in ("../../data/", "data/", ""):
        if os.path.exists(base + fname):
            return base + fname
    return None

# Tables this lesson needs — loaded from Training/data/, with a tiny built-in fallback.
FALLBACK = {
    'coffee_orders': [{'order_id': 'O-5001', 'date': '2026-03-07', 'item': 'Cappuccino', 'size': 'S', 'category': 'Espresso Drink', 'price': 3.75, 'payment': 'Cash', 'store': 'Downtown'}, {'order_id': 'O-5002', 'date': '2026-03-07', 'item': 'Mocha', 'size': 'S', 'category': 'Espresso Drink', 'price': 4.5, 'payment': 'Card', 'store': 'Airport'}, {'order_id': 'O-5003', 'date': '2026-03-05', 'item': 'Cappuccino', 'size': 'L', 'category': 'Espresso Drink', 'price': 5.25, 'payment': 'Card', 'store': 'Downtown'}, {'order_id': 'O-5004', 'date': '2026-03-05', 'item': 'Croissant', 'size': 'M', 'category': 'Food', 'price': 3.25, 'payment': 'App', 'store': 'Uptown'}, {'order_id': 'O-5005', 'date': '2026-03-06', 'item': 'Latte', 'size': 'L', 'category': 'Espresso Drink', 'price': 5.5, 'payment': 'App', 'store': 'Downtown'}],
}
frames = {}
for _name, _rows in FALLBACK.items():
    _p = _find(_name + ".csv")
    frames[_name] = pd.read_csv(_p) if _p else pd.DataFrame(_rows)

if BACKEND == "duckdb":
    try:
        import duckdb
    except ImportError:
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"], check=True)
        import duckdb
    _con = duckdb.connect(":memory:")                 # a private, in-memory warehouse
    for _name, _df in frames.items():
        _con.register("_src_" + _name, _df)
        _con.execute(f"CREATE OR REPLACE TABLE {_name} AS SELECT * FROM _src_{_name}")
    def run_sql(sql: str) -> pd.DataFrame:
        """Run SQL and return the result as a pandas DataFrame."""
        return _con.execute(sql).df()
else:
    import snowflake.connector
    from snowflake.connector.pandas_tools import write_pandas
    _con = snowflake.connector.connect(
        account=os.environ["SNOWFLAKE_ACCOUNT"], user=os.environ["SNOWFLAKE_USER"],
        password=os.environ["SNOWFLAKE_PASSWORD"], warehouse=os.environ.get("SNOWFLAKE_WAREHOUSE"),
        database=os.environ.get("SNOWFLAKE_DATABASE"), schema=os.environ.get("SNOWFLAKE_SCHEMA"))
    for _name, _df in frames.items():
        write_pandas(_con, _df, _name.upper(), auto_create_table=True, overwrite=True, quote_identifiers=False)
    def run_sql(sql: str) -> pd.DataFrame:
        """Run SQL and return the result as a pandas DataFrame."""
        cur = _con.cursor(); cur.execute(sql); return cur.fetch_pandas_all()

print(f"✅ Ready. Backend = {BACKEND.upper()} · tables: {', '.join(frames)}")

### 1 · Aggregate the whole table  →  like `df["price"].sum()`

An **aggregate function** collapses a whole column into one number. `COUNT(*)` counts rows; `SUM`, `AVG`, `MIN`, `MAX` do the obvious. Name each result with **`AS`**.

In [ ]:
run_sql("""
    SELECT
        COUNT(*)      AS num_orders,
        SUM(price)    AS total_revenue,
        AVG(price)    AS avg_order,
        MIN(price)    AS cheapest,
        MAX(price)    AS priciest
    FROM coffee_orders
""")

**What just happened:** five aggregates over the whole table in one query — the SQL twin of `df["price"].sum()`, `.mean()`, etc., bundled into a single result row. `COUNT(*)` counts every row; `COUNT(col)` would count only rows where `col` isn't null.

### 2 · `GROUP BY` — one summary row per category  →  like `df.groupby(...)`

The magic verb. **`GROUP BY category`** splits the table into groups, then the aggregate runs *once per group*. This is exactly `df.groupby("category")["price"].sum()`.

In [ ]:
# total revenue per category  (pandas: df.groupby("category")["price"].sum())
run_sql("""
    SELECT category, SUM(price) AS revenue
    FROM coffee_orders
    GROUP BY category
    ORDER BY revenue DESC
""")

In [ ]:
# count of orders per store, plus the average order value
run_sql("""
    SELECT store,
           COUNT(*)                 AS num_orders,
           ROUND(AVG(price), 2)     AS avg_order
    FROM coffee_orders
    GROUP BY store
    ORDER BY num_orders DESC
""")

**What just happened:** `GROUP BY store` made one row per store; `COUNT(*)` and `AVG(price)` were computed within each. **`ROUND(x, 2)`** trims the average to cents. Rule of thumb: **every column in `SELECT` that isn't wrapped in an aggregate must appear in `GROUP BY`.**

### 3 · `HAVING` — filter the *groups*  →  (there's no direct pandas twin)

`WHERE` filters **rows** *before* grouping. **`HAVING`** filters **groups** *after* — using the aggregate. "Only stores that made more than \$12" is a `HAVING` question, because "made more than \$12" is about the group's `SUM`, which doesn't exist until after grouping.

In [ ]:
run_sql("""
    SELECT store, ROUND(SUM(price), 2) AS revenue
    FROM coffee_orders
    GROUP BY store
    HAVING SUM(price) > 12          -- keep only the busy stores
    ORDER BY revenue DESC
""")

**What just happened:** every store was grouped and summed, then `HAVING` dropped the ones under \$12. If you'd written `WHERE SUM(price) > 12` you'd get an error — `WHERE` runs *before* the sum exists.

### 4 · The full pattern — `WHERE` → `GROUP BY` → `HAVING` → `ORDER BY`

Real reports stack the clauses. Read this one as a sentence: *"Among espresso drinks, total and average revenue per store, only stores above \$8, biggest first."*

In [ ]:
run_sql("""
    SELECT store,
           ROUND(SUM(price), 2)  AS revenue,
           ROUND(AVG(price), 2)  AS avg_order,
           COUNT(*)              AS num_orders
    FROM coffee_orders
    WHERE category = 'Espresso Drink'    -- 1. filter rows first
    GROUP BY store                        -- 2. group what's left
    HAVING SUM(price) > 8                 -- 3. filter the groups
    ORDER BY revenue DESC                 -- 4. sort the result
""")

> **The clause order that never changes:**
> `SELECT` → `FROM` → `WHERE` → `GROUP BY` → `HAVING` → `ORDER BY` → `LIMIT`.
> SQL always applies them in that order, even though you *read* `SELECT` first.

> **`Go Deeper 🔧` — `COUNT(DISTINCT ...)` and grouping by two columns.**
> - **`COUNT(DISTINCT store)`** counts unique values (how many stores appear), not rows.
> - **`GROUP BY store, category`** makes one row per *combination* — a mini pivot table.

In [ ]:
run_sql("""
    SELECT store,
           category,
           COUNT(*)              AS orders,
           COUNT(DISTINCT item)  AS distinct_items,
           ROUND(SUM(price), 2)  AS revenue
    FROM coffee_orders
    GROUP BY store, category
    ORDER BY store, revenue DESC
""")

> **`Common pitfalls ⚠️`**
>
> - **Non-aggregated columns must be in `GROUP BY`.** `SELECT store, item, SUM(price) ... GROUP BY store` errors — either group by `item` too or wrap it in an aggregate.
> - **`WHERE` filters rows, `HAVING` filters groups.** Condition uses a raw column → `WHERE`; condition uses `SUM`/`AVG`/`COUNT` → `HAVING`.
> - **`COUNT(*)` vs `COUNT(col)`:** `*` counts all rows; `COUNT(col)` skips nulls in that column.

### ✍️ Your turn

Each `run_sql(...)` returns a DataFrame you'll see immediately.

In [ ]:
# TODO 1: total revenue and order count per PAYMENT method (Cash / Card / App)

# TODO 2: the AVERAGE price per category, rounded to 2 decimals, priciest category first

# TODO 3: which items appear MORE THAN ONCE?  (GROUP BY item, HAVING COUNT(*) > 1)


<details><summary>✅ Show solution</summary>

```python
# 1
run_sql("""
    SELECT payment, COUNT(*) AS orders, ROUND(SUM(price), 2) AS revenue
    FROM coffee_orders
    GROUP BY payment
    ORDER BY revenue DESC
""")

# 2
run_sql("""
    SELECT category, ROUND(AVG(price), 2) AS avg_price
    FROM coffee_orders
    GROUP BY category
    ORDER BY avg_price DESC
""")

# 3
run_sql("""
    SELECT item, COUNT(*) AS times_ordered
    FROM coffee_orders
    GROUP BY item
    HAVING COUNT(*) > 1
    ORDER BY times_ordered DESC
""")
```
</details>

### 🚀 Build the artifact — a revenue-by-category report

One function, one query: the daily sales summary the shop owner actually wants. SQL groups and totals in the warehouse; you get a tidy DataFrame back.

In [ ]:
def revenue_report(min_revenue: float = 0.0) -> pd.DataFrame:
    """Revenue, order count, and average order value per category."""
    return run_sql(f"""
        SELECT category,
               COUNT(*)              AS num_orders,
               ROUND(SUM(price), 2)  AS revenue,
               ROUND(AVG(price), 2)  AS avg_order
        FROM coffee_orders
        GROUP BY category
        HAVING SUM(price) >= {min_revenue}
        ORDER BY revenue DESC
    """)

report = revenue_report()
print(f"Total across all categories: ${report['revenue'].sum():.2f}")
report

> **🔗 Your world — from coffee to matters.** This report *is* a billing summary. Swap the table and columns:
>
> ```sql
> SELECT practice_area,
>        COUNT(*)               AS num_matters,
>        ROUND(SUM(amount_billed), 2) AS total_billed,
>        ROUND(AVG(amount_billed), 2) AS avg_matter
> FROM matters
> WHERE status = 'Active'
> GROUP BY practice_area
> HAVING SUM(amount_billed) > 50000
> ORDER BY total_billed DESC;
> ```
>
> *Total billings by practice area, only the big ones, biggest first* — the exact query behind a partner's revenue dashboard.

### 📝 Recap — what you shipped

- **Aggregate functions** (`COUNT`, `SUM`, `AVG`, `MIN`, `MAX`) collapse a column into one number.
- **`GROUP BY`** computes an aggregate *per category* — the SQL twin of pandas `groupby`.
- **`AS`** names results; **`ROUND`** tidies them.
- **`HAVING`** filters groups by their aggregate; **`WHERE`** filters rows before grouping.
- **Artifact:** `revenue_report()` — a reusable group-and-total report returning a DataFrame.

### 🧠 Check your understanding

1. Which clause filters **groups**, and how does it differ from `WHERE`?
2. `SELECT store, item, SUM(price) FROM coffee_orders GROUP BY store` — why does this error?
3. What's the difference between `COUNT(*)` and `COUNT(item)`?
4. In what order does SQL apply `WHERE`, `GROUP BY`, `HAVING`, and `ORDER BY`?

<details><summary>✅ Answers</summary>

1. **`HAVING`** filters groups using an aggregate (e.g. `SUM(price) > 12`); **`WHERE`** filters individual rows *before* grouping.
2. `item` is neither grouped nor aggregated — every non-aggregated `SELECT` column must appear in `GROUP BY` (add `item`, or wrap it like `MAX(item)`).
3. `COUNT(*)` counts all rows; `COUNT(item)` counts only rows where `item` is not null.
4. **`WHERE` → `GROUP BY` → `HAVING` → `ORDER BY`.**
</details>

### ➡️ Next up — Week 3, Day 3: `JOIN` (stitching tables together)

Right now `coffee_orders` knows the *item* but not its *unit cost* (that's in a `menu` table) or the store's *region* (in a `stores` table). Next lesson we **join** tables on a shared key to compute real **profit** — the SQL twin of pandas `merge`, and the legal twin of *matching clients to their matters*.

*Nothing to install — the same `run_sql` helper carries over.*

### 📖 Reference & glossary

| Term | Plain meaning | pandas twin |
|---|---|---|
| **Aggregate function** | collapses a column to one value | `.sum()`, `.mean()`, ... |
| **`COUNT(*)`** | number of rows | `len(df)` |
| **`SUM` / `AVG` / `MIN` / `MAX`** | column total / average / smallest / largest | same-named methods |
| **`GROUP BY`** | one summary row per category | `df.groupby(col)` |
| **`HAVING`** | filter groups by their aggregate | `.groupby(...).filter(...)` |
| **`AS`** | name a result column | `.rename(...)` |
| **`ROUND(x, n)`** | round to *n* decimals | `.round(n)` |
| **`COUNT(DISTINCT c)`** | number of unique values | `df[c].nunique()` |

**Docs:** Snowflake aggregate functions — https://docs.snowflake.com/en/sql-reference/functions-aggregation

> *Not legal advice — these lessons teach technology. A lawyer reviews any AI or data output that will be relied upon.*